# tanque-cheio · 06 — teste final v2 (M17)

Roda os **6 sistemas da v2** no `datasets/test.jsonl` (150 perguntas), com as regras da
v2 para todos: prompt único, decodificação gulosa, `max_new_tokens = 384` (D31) e
autocorreção com 1 nova tentativa (D32). Cada sistema grava, numa só execução,
`<sistema>__test.jsonl` (single-shot), `<sistema>+retry__test.jsonl` e um
`run-info__...json` para cada (commit do checkpoint e do adapter, `quantization`,
`gpus`, `peak_gpu_mem_gib`, tempo).

O teste roda **uma vez por sistema** (regra 0.6, D33). O base e o ft-v1 já passaram por
ele na v1 (256 tokens, sem nova tentativa); aqui são reexecutados com as regras da v2, e
os números novos substituem os da v1 no relatório.

São **dois kernels** do mesmo notebook, escolhidos por `PART` na 1ª célula (D27, D37):

| `PART` | Kernel | GPUs | Sistemas |
|---|---|---|---|
| `"a"` | `belasoares/tanque-cheio-final-v2a` | 1 T4 (`CUDA_VISIBLE_DEVICES=0`) | `base-qwen2.5-coder-1.5b-instruct` (fp16), `ft-qwen2.5-coder-1.5b-instruct` (adapter v1, fundido), `ft2-qwen2.5-coder-1.5b-instruct` (adapter v2, fundido), `base-qwen2.5-coder-7b-instruct` e `base-qwen2.5-coder-14b-instruct` (nf4) |
| `"b"` | `belasoares/tanque-cheio-final-v2b` | 2 T4 (`device_map="auto"`) | `base-qwen2.5-coder-32b-instruct` (nf4), com limite de tempo e checkpoint por item |

Os checkpoints são os mesmos do M14 (`05_large_baselines.ipynb`), fixados por commit. O
adapter v2 (`belasoares/tanque-cheio-qwen2.5-coder-1.5b-lora-v2`) também vai por commit:
**`FT2_ADAPTER_REVISION` precisa ser preenchido** antes do push do kernel (a); sem ele,
a 2ª célula para com uma mensagem clara, antes de instalar qualquer coisa.

**Defesas (regra 0.8 e D37):** no kernel (a), `CUDA_VISIBLE_DEVICES = "0"` antes de
importar o torch; no (b), não (o 32B precisa das 2 T4). O `torchao` é removido e
`is_torchao_available()` é checado. Entre um modelo e outro, a memória da GPU e o cache
do Hugging Face são liberados. Se um modelo não puder rodar (falta de memória, de disco
ou de GPU), ele é pulado, o run-info registra o motivo e os outros seguem.

**No Kaggle:** *Settings → Accelerator → GPU T4 x2* e *Internet on* nos dois kernels (o
(a) usa só a 1ª GPU). Passos 👤 e estimativa de tempo em `notebooks/README.md`, seção
"Teste final v2 (M17)".

As métricas não são calculadas aqui: as previsões vão para o repositório e o CI recalcula
tudo (`tanque eval check-metrics`).

In [ ]:
# Parâmetros: edite só esta célula.
REPO_REF = "main"  # branch, tag ou commit do tanque-cheio
DATASET = "test"  # M17: o teste, uma vez por sistema (regra 0.6, D33)
LIMIT = None  # ex.: 3 para um ensaio rápido (os arquivos ganham o sufixo -sample3)
MAX_NEW_TOKENS = 384  # D31 (na v1 eram 256)
RETRY = True  # D32: grava também <sistema>+retry__<split>.jsonl
# "a" = kernel tanque-cheio-final-v2a (1 T4: 1,5B base, ft-v1, ft-v2, 7B, 14B);
# "b" = kernel tanque-cheio-final-v2b (2 T4: 32B). Troque antes de cada push.
PART = "a"
# Commit do adapter v2 no HF (o que `hf upload` imprimiu no M15, 40 caracteres
# hexadecimais). A Isabela preenche depois de publicar; com o placeholder, o kernel (a)
# para na 2ª célula.
FT2_ADAPTER_REVISION = "PREENCHER_COM_O_COMMIT_DO_ADAPTER_V2"
TIME_BUDGET_32B_H = 4  # D37: o 32B para depois disso (download e carga inclusos)
# Saída de uma versão anterior anexada como input (Add Input → Notebook Output), para
# continuar de onde parou: ex. "/kaggle/input/tanque-cheio-final-v2a". None = do zero.
RESUME_DIR = None
DTYPE = "float16"  # a T4 não tem bf16 nativo; também é o dtype de computação do nf4
OUT_DIR = "/kaggle/working"  # vira a saída do notebook no Kaggle
REPO_DIR = "/tmp/tanque-cheio"  # fora de OUT_DIR, para não ir para a saída

QWEN_15B = ("Qwen/Qwen2.5-Coder-1.5B-Instruct", "2e1fd397ee46e1388853d2af2c993145b0f1098a")

# Checkpoints fixados por commit: os mesmos do M14 (05_large_baselines.ipynb).
MODELS = [
    {
        "model_id": QWEN_15B[0],
        "revision": QWEN_15B[1],
        "quantization": None,  # fp16, como na v1
        "gpus": 1,
        "part": "a",
    },
    {
        "model_id": QWEN_15B[0],
        "revision": QWEN_15B[1],
        "adapter_id": "belasoares/tanque-cheio-qwen2.5-coder-1.5b-lora",  # ft-v1 (M6)
        "adapter_revision": "bd8079ff5467c339586bc7fc31f18a19b6509a9e",
        "quantization": None,  # fp16, adapter fundido
        "gpus": 1,
        "part": "a",
    },
    {
        "model_id": QWEN_15B[0],
        "revision": QWEN_15B[1],
        "adapter_id": "belasoares/tanque-cheio-qwen2.5-coder-1.5b-lora-v2",  # ft-v2 (M15)
        "adapter_revision": FT2_ADAPTER_REVISION,
        "system": "ft2-qwen2.5-coder-1.5b-instruct",  # 5.3 v2
        "quantization": None,  # fp16, adapter fundido
        "gpus": 1,
        "part": "a",
    },
    {
        "model_id": "Qwen/Qwen2.5-Coder-7B-Instruct",
        "revision": "c03e6d358207e414f1eca0bb1891e29f1db0e242",
        "quantization": "nf4",
        "gpus": 1,
        "part": "a",
    },
    {
        "model_id": "Qwen/Qwen2.5-Coder-14B-Instruct",
        "revision": "aedcc2d42b622764e023cf882b6652e646b95671",
        "quantization": "nf4",
        "gpus": 1,
        "part": "a",
    },
    {
        "model_id": "Qwen/Qwen2.5-Coder-32B-Instruct",
        "revision": "381fc969f78efac66bc87ff7ddeadb7e73c218a7",
        "quantization": "nf4",
        "gpus": 2,
        "max_memory": {0: "13GiB", 1: "14GiB"},  # folga na GPU 0 para ativações
        "part": "b",
        "time_budget_h": TIME_BUDGET_32B_H,
        # Só se o disco não comportar os 61 GiB em bf16 (no M14 não foi preciso).
        "fallback": {
            "model_id": "unsloth/Qwen2.5-Coder-32B-Instruct-bnb-4bit",
            "revision": "89e52be6bf3252ffa41a6db933aeba59230848dd",
        },
    },
]

In [ ]:
# Falha cedo (antes de instalar): PART válida e adapter v2 fixado por commit.
# No kernel (a), restringe a sessão a 1 GPU ANTES de importar o torch (regra 0.8): os 5
# sistemas rodam na mesma T4, como o 1,5B do M7. No (b), o 32B precisa das 2.
import os
import re

if PART not in ("a", "b"):
    raise ValueError(f'PART deve ser "a" (kernel final-v2a) ou "b" (final-v2b), não {PART!r}')
if PART == "a" and not re.fullmatch(r"[0-9a-f]{40}", str(FT2_ADAPTER_REVISION)):
    raise ValueError(
        "FT2_ADAPTER_REVISION não foi preenchido (vale "
        f"{FT2_ADAPTER_REVISION!r}). Ponha na 1ª célula o commit do adapter v2 no HF "
        "(belasoares/tanque-cheio-qwen2.5-coder-1.5b-lora-v2, 40 caracteres hexadecimais: "
        "o que o `hf upload` imprimiu, ou o da aba 'Files and versions') e faça o push de novo."
    )
if PART == "a":
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(f"PART={PART!r}; CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')!r}")

In [ ]:
# Instala o pacote `tanque` no REPO_REF (com as versões do M14) e clona o repositório no
# mesmo ref, para ter datasets/, configs/ e snapshot/.
import subprocess
import sys

REPO_URL = "https://github.com/belasoaress/tanque-cheio"
# As mesmas versões do M14 e do fine-tuning (uv.lock); torch vem da imagem.
PINS = [
    "transformers==5.21.0",
    "peft==0.21.2",
    "accelerate==1.15.0",
    "bitsandbytes==0.50.2",  # 4 bits (nf4); só existe com GPU
]


def sh(*cmd: str) -> None:
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


sh(sys.executable, "-m", "pip", "install", "-q", f"tanque @ git+{REPO_URL}@{REPO_REF}", *PINS)
# A imagem do Kaggle traz um torchao antigo (0.10) e o peft se recusa a carregar o
# adapter quando ele está instalado. O projeto não usa torchao: remove.
sh(sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao")
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "origin", REPO_REF)
sh("git", "-C", REPO_DIR, "checkout", "--quiet", "FETCH_HEAD")
REPO_COMMIT = subprocess.check_output(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True
).strip()
os.environ["TANQUE_ROOT"] = REPO_DIR  # antes de importar `tanque`
print("tanque-cheio em", REPO_COMMIT)

In [ ]:
import torch
import transformers
from peft.import_utils import is_torchao_available

print("torch", torch.__version__, "| transformers", transformers.__version__)
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(N_GPUS)]
print("GPUs visíveis:", N_GPUS, GPU_NAMES)
if PART == "a" and N_GPUS != 1:
    raise RuntimeError(
        f"Kernel (a) precisa de exatamente 1 GPU visível, há {N_GPUS}. Ligue o acelerador "
        "(Settings > Accelerator > GPU T4 x2) e confira CUDA_VISIBLE_DEVICES na 2ª célula."
    )
if PART == "b" and N_GPUS < 2:
    print(
        "Aviso: menos de 2 GPUs. Escolha Settings > Accelerator > GPU T4 x2; "
        "sem isso o 32B é pulado (D37)."
    )

# Falha cedo (antes da inferência) se o peft não puder carregar adapters.
is_torchao_available()  # levanta ImportError se houver um torchao incompatível
if N_GPUS:
    import bitsandbytes

    print("bitsandbytes", bitsandbytes.__version__)

In [ ]:
# Banco DuckDB do snapshot versionado (D25), para a autocorreção executar as SQLs
# (~6 s, sem download). Equivale a `tanque data build --from-parquet ...`.
sh(
    sys.executable,
    "-c",
    "from tanque.cli import app; app()",
    "data",
    "build",
    "--from-parquet",
    os.path.join(REPO_DIR, "snapshot", "coletas-2026-09.parquet"),
)

In [ ]:
# Continuação de uma versão anterior (opcional): copia os JSONL parciais para OUT_DIR;
# run_predictions(resume=True) pula os itens que já estão lá.
import shutil
from pathlib import Path

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
if RESUME_DIR:
    for src in sorted(Path(RESUME_DIR).rglob("*.jsonl")):
        dst = Path(OUT_DIR) / src.name
        if not dst.exists():
            shutil.copy2(src, dst)
            print("retomando de", src)

In [ ]:
import gc
import json

from tanque import config
from tanque.infer import large

selected = large.select_models(MODELS, PART)
large.require_pinned(selected)  # regra 0.8: nada roda no teste sem commit fixado
print(f"PART={PART!r}: {[large.system_of(m) for m in selected]}")

runs = []
for order, spec in enumerate(selected, start=1):
    info = large.run_spec(
        spec,
        dataset_path=config.resolve_path(f"datasets/{DATASET}.jsonl"),
        split=DATASET,
        out_dir=OUT_DIR,
        limit=LIMIT,
        max_new_tokens=MAX_NEW_TOKENS,
        retry=RETRY,
        dtype=DTYPE,
    )
    written = large.write_run_infos(
        info,
        OUT_DIR,
        repo_ref=REPO_REF,
        repo_commit=REPO_COMMIT,
        part=PART,
        kernel=f"belasoares/tanque-cheio-final-v2{PART}",
        session_order=order,
        gpu_names=GPU_NAMES,
    )
    # Só o resumo: o teste é sagrado, as SQLs não precisam aparecer no log.
    print(
        json.dumps(
            {k: v for k, v in info.items() if k not in ("summary", "retry_summary")},
            ensure_ascii=False,
        )
    )
    print("run-info:", [p.name for p in written])
    runs.append(info)
    # Libera a GPU antes do próximo modelo (run_predictions já apagou o modelo).
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# Resumo da sessão: o que rodou, o que parou (D37), o pico de memória e o tempo.
for info in runs:
    if info.get("skipped"):
        print(f"{info['system']}: PULADO ({info['skip_reason']})")
        continue
    s = info["summary"]
    r = info.get("retry_summary") or {}
    print(
        f"{info['system']}: {info['completed']}/{info['n_items']} itens"
        + (f" (parou: {info['stopped_reason']})" if info.get("stopped_reason") else "")
        + f", {info.get('quantization') or info['dtype']}, {info['gpus']} GPU(s), "
        f"pico {info['peak_gpu_mem_gib']} GiB, p50 {s['p50_latency_ms']} ms, "
        f"{r.get('retried', 0)} novas tentativas, {info['total_s'] / 60:.1f} min "
        f"[{info.get('model_source')}]"
    )
print("Arquivos em", OUT_DIR, sorted(p.name for p in Path(OUT_DIR).glob("*__*.json*")))